In [ ]:
import sys
import os

current_dir = os.getcwd()
if 'data_preprocessing' in current_dir or 'tests' in current_dir:
    src_dir = os.path.dirname(current_dir)
else:
    src_dir = os.path.join(current_dir, 'samples', 'order_flow_ray', 'src')
sys.path.append(src_dir)

import polars as pl
from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline
from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory

In [ ]:
# 1. Create pipeline
config = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        normalization=BMLLNormalizer()
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir})
)

pipeline = Pipeline(config)

In [ ]:
# 2. Run pipeline
results = pipeline.run(max_files=5)
print(f"Processed {len(results)} files")

In [ ]:
# 3. Inspect paths
result = results[0]
input_file = result['input_path']
output_file = result['output_path']

print(f"Input:  {input_file}")
print(f"Output: {output_file}")
print(f"Rows:   {result['row_count']:,}")

assert input_file != output_file
assert not output_file.startswith(config.data.raw_data_path)
print("✓ Paths validated")

In [ ]:
# 4. Read output (100 records)
data_access = DataAccessFactory.create('s3', region='us-east-1')
output_data = data_access.read(output_file).limit(100).collect()
print(f"✓ Read {len(output_data)} records")
output_data.head()

In [ ]:
# 5. Compare schemas
normalizer = BMLLNormalizer()
expected_schema = normalizer.get_schema('trades')

missing = set(expected_schema.keys()) - set(output_data.columns)
assert not missing, f"Missing columns: {missing}"

for col, expected_type in expected_schema.items():
    actual_type = output_data.schema[col]
    assert actual_type == expected_type, f"{col}: expected {expected_type}, got {actual_type}"

print(f"✓ Schema validated ({len(output_data.columns)} columns)")
print("\n✓ All tests passed!")